# Flanker Analysis in Nilearn — Part 1: First-Level (Within-Run) Analysis

This notebook recreates the [**first-level analysis from Andy's Brain Book**](https://andysbrainbook.readthedocs.io/en/latest/fMRI_Short_Course/fMRI_05_1stLevelAnalysis.html) (the FSL/FEAT flanker tutorial) using **Nilearn** and the preprocessed data from **fMRIPrep**.

For additional reference, be sure to check out the [Nilearn docs & demos](https://nilearn.github.io/dev/auto_examples/04_glm_first_level/index.html)

**What you will do:** for each of 3 subjects × 2 runs, build a GLM with two task regressors (*congruent*, *incongruent*) plus nuisance regressors, then estimate four contrasts:

| Contrast | Weights (incongruent, congruent) | Meaning |
|---|---|---|
| `incongruent` | [1, 0] | incongruent vs. baseline |
| `congruent` | [0, 1] | congruent vs. baseline |
| `incongruent-congruent` | [1, −1] | the flanker (conflict) effect |
| `all-baseline` | [0.5, 0.5] | average task activation vs. baseline |

**How to work through it:** read each text cell, run the code cell below it, and look at the output. You do not need to edit any code except the three paths at the start. Some steps take a while to run; the notebook tells you when, so you can work on the reflection questions in the meantime. *Remember to use your `fmri_class` conda environment with Nilearn already installed!*

**What to submit:** answers to the 3 reflection questions at the end of each of the three jupyter notebook files, in a separate Word or text file, uploaded to Canvas.

**You need:** `nilearn` (0.10 or newer), `nibabel`, `pandas`, `numpy`, `matplotlib`.

## Step 0 — Tell the notebook where your data are

In [ ]:
# ---- EDIT THESE THREE PATHS, THEN RUN THE REST OF THE NOTEBOOK UNCHANGED ----
BIDS_DIR     = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/rawdata"                       # contains sub-01/func/*_events.tsv
FMRIPREP_DIR = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/fmriprep_outputs/"  # contains sub-01/func/*_desc-preproc_bold.nii.gz
OUTPUT_DIR   = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/nilearn_outputs"   # where YOUR results will be saved

## Step 1 — Imports, settings, and file-location helpers
This cell defines the settings and small helper functions used throughout. It is worth skimming the comments so you know where each file comes from.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import nilearn

BIDS_DIR, FMRIPREP_DIR, OUTPUT_DIR = Path(BIDS_DIR), Path(FMRIPREP_DIR), Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Fixed settings for this assignment (you do not need to change these) ----
SUBS = ["sub-01", "sub-02", "sub-03"]
RUNS = ["run-1", "run-2"]
SPACE = "space-MNI152NLin2009cAsym_res-2"

# Contrast name -> {design-matrix column: weight}.  The 3 contrasts from the Andy's brain book demo, plus all>baseline.
# (all>baseline uses 0.5/0.5 so the result is the AVERAGE of the two conditions.)

CONTRASTS = {
    "incongruent":           {"incongruent": 1.0},
    "congruent":             {"congruent": 1.0},
    "incongruent-congruent": {"incongruent": 1.0, "congruent": -1.0},
    "all-baseline":          {"incongruent": 0.5, "congruent": 0.5},
}

# ---- Where files live (layout inside each folder follows BIDS / fMRIPrep) ----
def events_path(sub, run):
    return BIDS_DIR / sub / "func" / f"{sub}_task-flanker_{run}_events.tsv"

def fmriprep_path(sub, run, suffix):
    return FMRIPREP_DIR / sub / "func" / f"{sub}_task-flanker_{run}_{suffix}"

def bold_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-preproc_bold.nii.gz")
def mask_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-brain_mask.nii.gz")
def confounds_path(sub, run): return fmriprep_path(sub, run, "desc-confounds_timeseries.tsv")

# ---- Where this assignment writes its outputs ----
STAT_KEYS = {"effect": "effect_size", "variance": "effect_variance", "z": "z_score"}

def level1_path(sub, run, contrast, stat):
    return OUTPUT_DIR / "level1" / sub / f"{sub}_{run}_contrast-{contrast}_{stat}.nii.gz"

def level2_path(sub, contrast, stat):
    return OUTPUT_DIR / "level2" / sub / f"{sub}_contrast-{contrast}_{stat}.nii.gz"

def level3_path(contrast, smoothing, stat):
    return OUTPUT_DIR / "level3" / f"group_contrast-{contrast}_smooth-{smoothing}_{stat}.nii.gz"


from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix, spm_hrf
from nilearn.plotting import plot_design_matrix, plot_stat_map

# ---- Model settings ----
FD_THRESHOLD = 0.5      # framewise displacement (mm) above which a volume gets its own "spike" regressor
HIGH_PASS = 1 / 100     # high-pass cutoff in Hz (100 s period, same as the FSL tutorial)
SLICE_TIME_REF = 0.5    # fMRIPrep slice-time-corrects to the middle of the TR
N_COMPCOR = 5           # number of aCompCor components to include
MOTION = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
ACOMPCOR = [f"a_comp_cor_{i:02d}" for i in range(N_COMPCOR)]

In [ ]:
# Check that every file we need is where we expect, and say clearly what is missing
needed = []
for sub in SUBS:
    for run in RUNS:
        needed += [events_path(sub, run), bold_path(sub, run), mask_path(sub, run), confounds_path(sub, run)]
missing = [p for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError("These files were not found - check your paths above:\n" + "\n".join(str(p) for p in missing[:10]))
print(f"Found all {len(needed)} input files.")

### How does this differ from the FSL analysis? (conceptual)

| Step | FSL / FEAT (Andy's Brain Book) | Nilearn (this notebook) |
|---|---|---|
| Hemodynamic response | Double-gamma HRF | SPM canonical HRF — also a double gamma, very similar but not identical parameters |
| Autocorrelation | FILM prewhitening (autocorrelation estimates smoothed across space) | AR(1) model fit to the residuals; voxels with similar AR(1) coefficients share a whitening filter |
| High-pass filter | Filters data and model with a 100 s cutoff | Adds slow *cosine drift regressors* (100 s period) to the design matrix |
| Spatial smoothing | none (although in some of Andy's brain book tutorials there is 5mm smoothing) | **None** at this level |
| Signal scaling | Optional intensity normalization | Each voxel scaled to **percent of its mean** by default, so effect sizes are in % signal change |
| Outputs | `cope`, `varcope`, `zstat` | `effect_size`, `effect_variance`, `z_score` |
| Nuisance regressors | Motion/confound text file added as EVs | Motion, aCompCor, and spike regressors added as columns of the design matrix |

## Step 2 — Read the task events

Each run has a BIDS `events.tsv` file listing when each trial started (`onset`), how long it lasted (`duration`), and what kind of trial it was (`Stimulus`). We keep only the **congruent** and **incongruent** trials and rename the column to `trial_type`, which is what Nilearn expects.

In [ ]:
def load_events(sub, run):
    ev = pd.read_csv(events_path(sub, run), sep="\t")
    ev = ev.loc[ev["Stimulus"].isin(["congruent", "incongruent"]), ["onset", "duration", "Stimulus"]]
    return ev.rename(columns={"Stimulus": "trial_type"}).reset_index(drop=True)

# a giant dictionary with the events file for each individual run inside it
EVENTS = {(sub, run): load_events(sub, run) for sub in SUBS for run in RUNS}

print("First few events from", SUBS[0], RUNS[0])
display(EVENTS[(SUBS[0], RUNS[0])].head(8))

counts = pd.DataFrame({f"{s} {r}": EVENTS[(s, r)]["trial_type"].value_counts()
                       for s in SUBS for r in RUNS}).T
print("Number of trials per condition in each run:")
display(counts)

## Step 3 — Build the nuisance regressors from fMRIPrep

fMRIPrep gives us a `confounds_timeseries.tsv` for each run. From it we take:

- **6 motion parameters** (3 translations, 3 rotations)
- **5 aCompCor components** (noise estimated from regions that shouldn't contain BOLD signal; we do *not* use the CSF, white matter, or global signal regressors)
- **Spike regressors:** for every volume whose **framewise displacement (FD) exceeds 0.5 mm**, we add a column that is 1 at that volume and 0 everywhere else. This effectively removes that volume's influence from the model **without deleting it**, so the timing of the task is left intact.

In [ ]:
def load_confounds(sub, run):
    raw = pd.read_csv(confounds_path(sub, run), sep="\t")
    conf = raw[MOTION + ACOMPCOR].fillna(0).copy()

    fd = raw["framewise_displacement"].fillna(0)          # first volume has no FD (NaN) -> 0
    spike_idx = np.where(fd.values > FD_THRESHOLD)[0]
    spikes = pd.DataFrame({f"spike_{i:03d}": (np.arange(len(raw)) == i).astype(float) for i in spike_idx},
                          index=raw.index)
    return pd.concat([conf, spikes], axis=1), fd, spike_idx

CONFOUNDS, FD, SPIKES = {}, {}, {}
for sub in SUBS:
    for run in RUNS:
        CONFOUNDS[(sub, run)], FD[(sub, run)], SPIKES[(sub, run)] = load_confounds(sub, run)

# Which volumes were flagged?
summary = pd.DataFrame([{"subject": s, "run": r,
                         "n_volumes": len(FD[(s, r)]),
                         "n_flagged (FD > 0.5 mm)": len(SPIKES[(s, r)]),
                         "max FD (mm)": round(FD[(s, r)].max(), 2),
                         "n_confound_columns": CONFOUNDS[(s, r)].shape[1]} for s in SUBS for r in RUNS])
display(summary)

fig, axes = plt.subplots(len(SUBS), len(RUNS), figsize=(11, 7), sharey=True, sharex=True)
for i, sub in enumerate(SUBS):
    for j, run in enumerate(RUNS):
        ax = axes[i, j]
        fd, sp = FD[(sub, run)], SPIKES[(sub, run)]
        ax.plot(fd.values, lw=1)
        ax.axhline(FD_THRESHOLD, color="r", ls="--", lw=1)
        ax.plot(sp, fd.values[sp], "ro", ms=4)
        ax.set_title(f"{sub} {run}", fontsize=10)
        if j == 0: ax.set_ylabel("FD (mm)")
        if i == len(SUBS) - 1: ax.set_xlabel("Volume")
fig.suptitle("Framewise displacement (red dashed line = 0.5 mm threshold; red dots = flagged volumes)")
plt.tight_layout()

## Step 4 — The hemodynamic response function (HRF)

The model assumes each trial triggers a BOLD response shaped like the HRF. Nilearn's `hrf_model="spm"` uses a **double-gamma** function: a positive peak around 5 s followed by a slower, shallower undershoot — the same basic shape FSL uses.

In [ ]:
tr = nib.load(str(bold_path(SUBS[0], RUNS[0]))).header.get_zooms()[3]
print(f"TR = {tr} s")

oversampling = 16
hrf = spm_hrf(tr, oversampling=oversampling)
t = np.arange(len(hrf)) * tr / oversampling

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(t, hrf, lw=2)
ax.axhline(0, color="gray", lw=0.5)
ax.set(xlabel="Time after trial onset (s)", ylabel="Response (a.u.)", title="SPM canonical (double-gamma) HRF")
plt.tight_layout()

## Step 5 — Look at the design matrix for every subject and run

Before fitting anything, build the **design matrix** and look at it. Each **column** is a regressor; each **row** is one fMRI volume (time runs top to bottom). You should be able to find:

- the two **task regressors** (`congruent`, `incongruent`): the stimulus onsets convolved with the HRF
- the slow **cosine drift** columns (`drift_1`, `drift_2`, ...) that act as the high-pass filter
- the 6 **motion** and 5 **aCompCor** columns
- the **spike regressors** (`spike_XXX`): look for columns that are zero everywhere except a single row — each one marks one flagged volume
- the `constant` (intercept) column at the far right

Compare each subject's two runs: do the number and positions of the spike columns match what you saw in the FD plots above?

In [ ]:
def build_design(sub, run):
    # One row per volume; assumes confounds have the same length as the BOLD run
    n = len(CONFOUNDS[(sub, run)])

    # Time (in seconds) at which each volume is treated as sampled:
    #   np.arange(n) * tr        -> volume onset times (0, TR, 2*TR, ...)
    #   + SLICE_TIME_REF * tr    -> shift to the slice-timing reference used by
    #                               preprocessing (fMRIPrep default = 0.5, mid-TR),
    #                               so events line up with the corrected data
    frame_times = np.arange(n) * tr + SLICE_TIME_REF * tr

    return make_first_level_design_matrix(
        frame_times,
        EVENTS[(sub, run)],           # onset / duration / trial_type table
        hrf_model="spm",              # convolve events with the SPM canonical HRF
        drift_model="cosine",         # cosine basis to model slow scanner drift
        high_pass=HIGH_PASS,          # drift cutoff in Hz (e.g. 1/128)
        add_regs=CONFOUNDS[(sub, run)],  # nuisance regressors (motion, CompCor, etc.); no NaNs allowed
    )

DESIGNS = {(sub, run): build_design(sub, run) for sub in SUBS for run in RUNS}

def draw_design(dm, ax):
    try:
        plot_design_matrix(dm, axes=ax)      # newer nilearn
    except TypeError:
        plot_design_matrix(dm, ax=ax)        # older nilearn

for sub in SUBS:
    fig, axes = plt.subplots(1, len(RUNS), figsize=(13, 6))
    for ax, run in zip(axes, RUNS):
        draw_design(DESIGNS[(sub, run)], ax)
        ax.set_title(f"{sub} {run}  ({DESIGNS[(sub, run)].shape[1]} columns)")
    plt.tight_layout()
    plt.show()

## Step 6 — Define the contrasts

A **contrast** is a weighted sum of the estimated betas. Nilearn wants one weight for *every* column of the design matrix, so we build a vector that is zero everywhere except the task regressors. The cell below prints the non-zero weights for each contrast.

In [ ]:
def contrast_vector(design_matrix, weights):
    vec = np.zeros(design_matrix.shape[1])
    for name, w in weights.items():
        vec[design_matrix.columns.get_loc(name)] = w
    return vec

example_dm = DESIGNS[(SUBS[0], RUNS[0])]
for cname, weights in CONTRASTS.items():
    vec = contrast_vector(example_dm, weights)
    nz = {example_dm.columns[i]: vec[i] for i in np.nonzero(vec)[0]}
    print(f"{cname:25s} non-zero weights: {nz}   (vector length = {len(vec)})")

### Now let's visualize the contrasts with a plot (compare this to the FSL design matrix plot!)

In [ ]:
import matplotlib.pyplot as plt

# Build a (n_contrasts x n_regressors) matrix of contrast weights
cnames = list(CONTRASTS.keys())
C = np.vstack([contrast_vector(example_dm, CONTRASTS[c]) for c in cnames])

# Optional: crop to columns used by at least one contrast, so the dozens of
# confound columns don't squash the plot. Set to True to show only the non-zero columns
ONLY_USED_COLUMNS = False
if ONLY_USED_COLUMNS:
    used = np.where(np.any(C != 0, axis=0))[0]
else:
    used = np.arange(C.shape[1])
C_plot = C[:, used]
col_labels = [example_dm.columns[i] for i in used]

fig, ax = plt.subplots(figsize=(max(6, 0.9 * len(used) + 3), 0.6 * len(cnames) + 2))

# Symmetric color limits so 0 is always the neutral (white) midpoint
vmax = np.abs(C_plot).max() or 1
im = ax.imshow(C_plot, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=60, ha="right")
ax.set_yticks(range(len(cnames)))
ax.set_yticklabels(cnames)

# Write the weight in each non-zero cell
for i in range(C_plot.shape[0]):
    for j in range(C_plot.shape[1]):
        if C_plot[i, j] != 0:
            ax.text(j, i, f"{C_plot[i, j]:g}", ha="center", va="center",
                    color="white" if abs(C_plot[i, j]) > 0.6 * vmax else "black")

# Light grid between cells
ax.set_xticks(np.arange(-.5, len(col_labels), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(cnames), 1), minor=True)
ax.grid(which="minor", color="lightgray", linewidth=0.5)
ax.tick_params(which="minor", length=0)

ax.set_title(f"Contrast weights (design from sub-{SUBS[0]}, run {RUNS[0]})")
fig.colorbar(im, ax=ax, label="weight", shrink=0.8)
plt.tight_layout()
plt.show()

## Step 7 — Fit the model for every run  (WARNING: could be slow!)

> **This is the slow step.** Each run is fit voxel by voxel across the whole brain, with an AR(1) noise model, so it can take a while for each run (6 runs total) depending on your laptop. **Start working on the reflection questions at the bottom of the notebook while this runs.**

For each run, the code below:
1. fits the GLM (Nilearn [FirstLevelModel](https://nilearn.github.io/dev/modules/generated/nilearn.glm.first_level.FirstLevelModel.html)) with **no smoothing**, the SPM double-gamma HRF, cosine drift (100 s), and an **AR(1) noise model** (this is Nilearn's prewhitening). **note:** the `FirstLevelModel` function can compute the design matrix internally given events and confounds files, so the code below checks that this is identical to the one you made manually above with `make_first_level_design_matrix`
2. computes the four contrasts
3. saves three maps for each: the **effect size** (the contrast estimate, in % signal change), its **variance**, and the **z-statistic**. Part 2 will need the effect size *and* variance maps.

If a run's outputs already exist (for example if you re-run the notebook), it is skipped.

In [ ]:
import time

# Loop over every subject x run: each run is fit as its own first-level GLM
for sub in SUBS:
    for run in RUNS:
        # Resume-friendly: if every contrast/stat map for this run is already on disk, skip it
        expected = [level1_path(sub, run, c, s) for c in CONTRASTS for s in STAT_KEYS]
        if all(p.exists() for p in expected):
            print(f"{sub} {run}: outputs already exist, skipping")
            continue

        start = time.time()

        # Set up the GLM. Timing/drift settings mirror build_design() so the model
        # nilearn builds internally should match the design matrix you inspected.
        #   noise_model="ar1"  -> accounts for temporal autocorrelation in the residuals
        #                         (prewhitening, like FSL's FILM), which keeps t-stats honest
        #   smoothing_fwhm=None -> no spatial smoothing applied at this stage
        #   mask_img           -> only fit voxels inside the brain mask
        #   minimize_memory    -> discard big intermediate arrays; we only need contrasts
        model = FirstLevelModel(t_r=tr, slice_time_ref=SLICE_TIME_REF,
                                hrf_model="spm", drift_model="cosine", high_pass=HIGH_PASS,
                                noise_model="ar1", smoothing_fwhm=None,
                                mask_img=str(mask_path(sub, run)), minimize_memory=True)

        # Fit the GLM voxelwise: BOLD data ~ events (convolved with HRF) + drift + confounds
        model.fit(str(bold_path(sub, run)), events=EVENTS[(sub, run)], confounds=CONFOUNDS[(sub, run)])

        # Sanity check: the design matrix nilearn built should equal the one from build_design().
        # If this prints False, the events, confounds, or timing settings differ somewhere.
        dm = model.design_matrices_[0]
        same = dm.shape == DESIGNS[(sub, run)].shape and np.allclose(dm.values, DESIGNS[(sub, run)].values, atol=1e-4)
        print(f"{sub} {run}: design matrix matches the one you inspected: {same}")

        # Make sure the output folder exists before writing into it
        level1_path(sub, run, "incongruent", "z").parent.mkdir(parents=True, exist_ok=True)

        # For each contrast: turn the weights dict into a vector aligned with the
        # design matrix columns, compute it, and save each requested statistic map.
        # output_type="all" returns a dict of maps (z, effect size, variance, etc.),
        # and STAT_KEYS maps our filename labels to the keys in that dict.
        # The effect size and variance maps (cope/varcope equivalents) are what
        # a second-level (group) analysis needs.
        for cname, weights in CONTRASTS.items():
            res = model.compute_contrast(contrast_vector(dm, weights), stat_type="t", output_type="all")
            for stat, key in STAT_KEYS.items():
                res[key].to_filename(str(level1_path(sub, run, cname, stat)))
        print(f"{sub} {run}: done in {(time.time() - start) / 60:.1f} min")

## Step 8 — Look at the results

Here is the **incongruent − congruent** z-map for every run. These maps are shown **without any statistical threshold** (we will deal with thresholding later), so the colors simply show how large the z-statistic is at each voxel — positive in red/yellow, negative in blue.

Things to notice: how similar are the two runs for the same subject? How similar are different subjects?

In [ ]:
fig, axes = plt.subplots(len(SUBS), len(RUNS), figsize=(14, 3.2 * len(SUBS)))
for i, sub in enumerate(SUBS):
    for j, run in enumerate(RUNS):
        plot_stat_map(str(level1_path(sub, run, "incongruent-congruent", "z")), axes=axes[i, j],
                      display_mode="z", cut_coords=[20, 35, 50], vmax=5,
                      title=f"{sub} {run}: incongruent - congruent (z)")
plt.show()

## Reflection questions

Write your answers in a **separate Word document or text file** (not in this notebook) and upload it to Canvas. A few sentences for each is enough.

1. **Spike regressors.** Look at the FD plots and the design matrices. Which run had the most flagged volumes? Explain what a spike regressor does to the model, and why this approach is preferable to deleting the flagged volumes from the data in a task fMRI experiment.

2. In what ways do you think you Nilearn offers advantages over FSL in running GLMs? In what ways are there disadvantages?

3. Let's say you wanted to check whether FSL and Nilearn GLMs were giving you the same results. How could you do this?